In [44]:
import pandas as pd
import numpy as np
customers = pd.read_csv("../../data/raw/customers.csv")
order_items = pd.read_csv("../../data/raw/order_items.csv")
orders = pd.read_csv("../../data/raw/orders.csv")
products = pd.read_csv("../../data/raw/products.csv")

print(customers.info())
print(order_items.info())
print(orders.info())
print(products.info())

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column

In [45]:
#날짜 타입 / 결측 확인 끝 rangeIndex가 300 이고 entries도 300이니 결측은 없다.
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
# coerce는 에러나면 null이 됨
orders["order_date"].info()
orders.head()

<class 'pandas.Series'>
RangeIndex: 300 entries, 0 to 299
Series name: order_date
Non-Null Count  Dtype         
--------------  -----         
300 non-null    datetime64[us]
dtypes: datetime64[us](1)
memory usage: 2.5 KB


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-02,card,completed
1,2,77,2025-09-17,naver_pay,cancelled
2,3,138,2026-01-14,bank_transfer,cancelled
3,4,57,2026-03-27,kakao_pay,cancelled
4,5,125,2026-02-15,card,cancelled


In [46]:
# 키 결측 확인 (부모 없는 자식이 있냐) 부모 테이블 중 결측 확인

print(customers["customer_id"].isna().sum())
print(order_items["order_item_id"].isna().sum())
print(orders["order_id"].isna().sum())
print(products["product_id"].isna().sum())

0
0
0
0


In [47]:
# 키 중복 확인
for frame, key in [(customers, "customer_id"),(order_items, "order_item_id"),(orders, "order_id"),(products, "product_id")]:
    print(key, "결측 개수: ", frame[key].isna().sum())

customer_id 결측 개수:  0
order_item_id 결측 개수:  0
order_id 결측 개수:  0
product_id 결측 개수:  0


In [48]:
order_items["total_price"] = order_items["quantity"] * order_items["unit_price"]
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,total_price
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 단변량 EDA

변수 하나의 분포와 상태를 확인한다.

In [49]:
# 도시(city), 성별(gender), 나이(age)로 등록 고객의 분포(value_count, describe)를 확인해 주세요.

customers["city"].value_counts()
customers["gender"].value_counts()

customers["age"].describe()
# 연속형은 describe()

count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

## 이변량 EDA

두 변수의 차이나 관계를 살펴 봅니다.

In [50]:
# 상품  카테고리의 개수와 가격 통계

product_count = products["category"].value_counts(dropna=False)
type(product_count)

pandas.Series

In [51]:
products["price"].describe()

count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [54]:
category_price =  products.groupby("category", dropna=False).agg(
    product_count = ("product_id", "size"),
    mean_price = ("price", "mean"),
    median_price = ("price", "median")
)

print(category_price)


          product_count     mean_price  median_price
category                                            
도서                   14  106857.142857      118500.0
뷰티                   16  117687.500000      134000.0
생활용품                 16   96437.500000       89000.0
스포츠                  19  111578.947368      103000.0
식품                    7  137142.857143      147000.0
전자기기                 17  101588.235294      111000.0
패션                   11  115909.090909      115000.0


In [ ]:
# 완료 주문 병합 후 검증

In [59]:
# orders["order_status"].value_counts()
completed_orders = orders[orders["order_status"] == "complited"]
print(completed_orders["order_status"].value_counts())

Series([], Name: count, dtype: int64)


In [62]:
# order_items 에서 completed인 항목
items_with_orders = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator=True,
)

items_with_orders.head()

,order_item_id,order_id,product_id,quantity,unit_price,total_price,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,306000,123,2026-07-02,completed,both
1,2,1,87,5,25000,125000,123,2026-07-02,completed,both
2,3,1,7,3,142000,426000,123,2026-07-02,completed,both
3,4,1,9,3,193000,579000,123,2026-07-02,completed,both
4,5,2,72,4,189000,756000,77,2025-09-17,cancelled,both


In [ ]:
# 고객별 평균 주문 건수(완료건 기준)

complited_items_with_orders = items_with_orders[items_with_orders["order_status"] == "completed"]  
print(complited_items_with_orders["order_status"].value_counts())

order_status
completed    474
Name: count, dtype: int64


In [69]:
customer_completed_orders = complited_items_with_orders.groupby("customer_id")["order_id"].nunique()

customer_completed_orders.head()

customer_id
3    2
4    1
5    2
6    2
7    1
Name: order_id, dtype: int64

In [74]:
# 카테고리별 수량 합산하기, 카테고리별 완료 주문 금액 합산

order_category = order_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
)

category_quantity = order_category.groupby("category")["quantity"].sum()
print(category_quantity)

completed_order_ids = orders.loc[orders["order_status"] == "completed", "order_id"]

completed_items = order_category[order_category["order_id"].isin(completed_order_ids)]

category_completed_amount = completed_items.groupby("category")["total_price"].sum()
print(category_completed_amount)

category
도서      238
뷰티      376
생활용품    390
스포츠     468
식품      240
전자기기    401
패션      220
Name: quantity, dtype: int64
category
도서      16389000
뷰티      23383000
생활용품    23915000
스포츠     31743000
식품      16573000
전자기기    26400000
패션      10587000
Name: total_price, dtype: int64
